In [ ]:
!pip install pyautogen google-generativeai pillow --quiet

import google.generativeai as genai
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os
import json
from PIL import Image
import base64
import io

# Configure API and model
class SystemConfig:
    def __init__(self, model_name: str = "gemini-pro"):
        self.api_key = os.getenv("GEMINI_API_KEY", "your_gemini_api_key")
        if not self.api_key or self.api_key == "your_gemini_api_key":
            raise EnvironmentError("GEMINI_API_KEY not set")
        genai.configure(api_key=self.api_key)
        self.llm_config = {
            "config_list": [
                {
                    "model": model_name,
                    "api_key": self.api_key,
                    "api_type": "google"
                }
            ],
            "temperature": 0.5,  # Changed from 0.7
            "max_tokens": 1000   # Added token limit
        }

# Agents
client_proxy = UserProxyAgent(
    name="Client_Proxy",
    human_input_mode="NEVER",
    code_execution_config={"use_docker": False},
    system_message="""You are the Client Proxy Agent.
    Initiate bill analysis by providing bill data or images.
    Represent user needs in the expense tracking process."""
)

expense_analyzer = AssistantAgent(
    name="Expense_Analyzer",
    llm_config=SystemConfig().llm_config,
    system_message="""You are the Expense Analysis Agent.
    Responsibilities:
    1. Extract data from bill inputs
    2. Categorize expenses into: food, dining, utilities, retail, leisure, travel, medical, misc
    3. List items and their costs
    4. Compute category totals

    Output Format:
    - Category: [Name]
    - Items: [Item: Price]
    - Total: [Amount]

    Provide a clear expense breakdown."""
)

budget_inspector = AssistantAgent(
    name="Budget_Inspector",
    llm_config=SystemConfig().llm_config,
    system_message="""You are the Budget Inspection Agent.
    Responsibilities:
    1. Review categorized expenses
    2. Calculate total spending by category
    3. Identify spending trends
    4. Flag high spending areas
    5. Offer budget recommendations

    Output Format:
    - Total Spending: [Amount]
    - Breakdown: [Category: Amount, Percent]
    - Top Category: [Name]
    - Budget Tips: [Insights]

    Provide actionable budget advice."""
)

chat_group = GroupChat(
    agents=[client_proxy, expense_analyzer, budget_inspector],
    messages=[],
    max_round=6,  # Changed from 10
    speaker_selection_method="round_robin"
)

chat_coordinator = GroupChatManager(
    groupchat=chat_group,
    llm_config=SystemConfig().llm_config,
    system_message="""You are the Chat Coordinator for the Expense Tracking System.
    Manage agent interactions for efficient bill processing.

    Workflow:
    1. Client Proxy submits bill data
    2. Expense Analyzer categorizes expenses
    3. Budget Inspector provides insights

    Ensure smooth agent collaboration."""
)

# Image processing
def handle_image(image_path: str) -> Optional[str]:
    """Convert image to base64 for processing."""
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            max_size = (800, 800)  # Changed from 1024x1024
            img.thumbnail(max_size, Image.Resampling.LANCZOS)
            buffer = io.BytesIO()
            img.save(buffer, format='PNG')  # Changed to PNG
            img_str = base64.b64encode(buffer.getvalue()).decode()
            return img_str
    except Exception as e:
        print(f"Image processing error: {e}")
        return None

# Simulated bill data
def generate_sample_bill(bill_type: str) -> Dict:
    """Generate sample bill data."""
    sample_data = {
        "supermarket": {
            "items": ["Cereal - $4.99", "Milk - $3.29", "Bananas - $2.50", "Cheese - $6.75"],
            "total": 17.53,
            "category": "food"
        },
        "cafe": {
            "items": ["Coffee - $5.00", "Sandwich - $8.50", "Pastry - $3.25", "Tip - $2.25"],
            "total": 19.00,
            "category": "dining"
        },
        "clothing": {
            "items": ["Shirt - $30.00", "Pants - $50.00", "Jacket - $90.00"],
            "total": 170.00,
            "category": "retail"
        }
    }
    return sample_data.get(bill_type.lower(), sample_data["supermarket"])

# Main function
def run_expense_tracker():
    print("=== Expense Tracking System ===")
    print("Process bills and gain spending insights.")

    print("\nOptions:")
    print("1. Upload bill image")
    print("2. Use sample data")
    choice = input("Select option (1 or 2): ").strip()

    bill_data = None
    if choice == "1":
        image_path = input("Enter image path: ").strip()
        if os.path.exists(image_path):
            bill_data = f"Bill image: {image_path}"
            print(f"✅ Image loaded: {image_path}")
        else:
            print("❌ Image not found. Using sample data.")
            bill_data = generate_sample_bill("supermarket")
    else:
        bill_type = input("Enter bill type (supermarket/cafe/clothing): ").strip()
        bill_data = generate_sample_bill(bill_type)
        print(f"✅ Using sample {bill_type} bill")

    initial_message = f"""
    Process this bill for expense tracking:

    Bill Data: {json.dumps(bill_data, indent=2)}

    Tasks:
    1. Categorize expenses
    2. Calculate category totals
    3. Provide budget insights

    Begin processing.
    """

    print("\n" + "="*40)
    print("🚀 Starting Expense Tracking...")
    print("="*40)

    try:
        result = client_proxy.initiate_chat(
            recipient=chat_coordinator,
            message=initial_message,
            max_turns=5  # Changed from 8
        )

        # Save output to JSON
        output = {
            "bill_data": bill_data,
            "analysis": expense_analyzer.last_message().get("content", "No analysis"),
            "insights": budget_inspector.last_message().get("content", "No insights")
        }
        with open("expense_report.json", "w") as f:
            json.dump(output, f, indent=2)

        print("\n" + "="*40)
        print("✅ Expense Tracking Complete!")
        print("="*40)
        print("\n📄 Report saved to expense_report.json")

        return output

    except Exception as e:
        print(f"System error: {e}")
        return None

# Simplified version
def basic_expense_report():
    print("=== Basic Expense Report ===")
    expenses = {
        "Food": 17.53,
        "Dining": 19.00,
        "Retail": 170.00,
        "Utilities": 75.25,
        "Leisure": 40.00
    }

    total = sum(expenses.values())
    top_category = max(expenses, key=expenses.get)

    output = {
        "total_spending": total,
        "breakdown": [
            {"category": k, "amount": v, "percent": (v / total) * 100}
            for k, v in expenses.items()
        ],
        "top_category": top_category,
        "insights": []
    }

    if expenses[top_category] > total * 0.5:  # Changed threshold from 0.4
        output["insights"].append(f"High spending in {top_category} - review budget")
    output["insights"].append(f"Weekly projection: ${total * 4:.2f}")
    output["insights"].append(f"Daily average: ${total / 7:.2f}")

    print(f"\n📊 Summary:")
    print(f"💰 Total: ${total:.2f}")
    print(f"\n📈 Breakdown:")
    for item in output["breakdown"]:
        print(f"  • {item['category']}: ${item['amount']:.2f} ({item['percent']:.1f}%)")
    print(f"\n🎯 Top Category: {top_category} (${expenses[top_category]:.2f})")
    print(f"\n💡 Insights:")
    for insight in output["insights"]:
        print(f"  • {insight}")

    with open("basic_expense_report.json", "w") as f:
        json.dump(output, f, indent=2)

if __name__ == "__main__":
    try:
        run_expense_tracker()
    except Exception as e:
        print(f"Multi-agent error: {e}")
        print("Switching to basic report...")
        basic_expense_report()